# 07 · Phase B: Kronos across 29 equities + ETH, Chronos comparison, synthetic mechanism test (Colab T4)

Order = most informative first; every step resumes after a disconnect (rerun **Setup**, then the step).
No secrets needed (no news / LLM in this notebook). Total ≈ 3–3.5 GPU-hours; it can span several sessions.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'
KRONOS = '/content/Kronos'

# Unpack code from Drive. Set UPDATE = True once after uploading a new kronos-agent.zip.
import os, shutil, zipfile, sys
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/kronos_agent/phaseb.py'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

!test -d $KRONOS || git clone -q https://github.com/shiyu-coder/Kronos $KRONOS
%pip install -q einops safetensors ccxt yfinance chronos-forecasting
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)

import glob
EQ = ' '.join(sorted(os.path.relpath(p, CODE) for p in glob.glob(f'{CODE}/configs/kronos_b/*.yaml')))
ETH = 'configs/kronos_b_crypto/eth.yaml'
CH = '--set model_family=chronos --set model_id=amazon/chronos-t5-small --set amp=false'
!nvidia-smi -L
print(len(EQ.split()), 'equity configs')

## Step 1 · Synthetic random walks with known drift (GPU, ~15 min) — does the reversion come from the model?

In [ ]:
!cd $CODE && python scripts/run_synthetic.py --root $ROOT --kronos-path $KRONOS --lookback 64 256
!cd $CODE && python scripts/run_synthetic.py --root $ROOT --model-family chronos --model-id amazon/chronos-t5-small --lookback 64 256

## Step 2 · Kronos, lookback 64, 29 equities (GPU, ~30 min)

In [ ]:
!cd $CODE && python scripts/run_sampling.py --root $ROOT --kronos-path $KRONOS --config $EQ --set lookback=64

## Step 3 · Chronos-T5-small, lookbacks 64 and 256 (GPU, ~45 min)

In [ ]:
!cd $CODE && python scripts/run_sampling.py --root $ROOT --config $EQ --set lookback=64 $CH
!cd $CODE && python scripts/run_sampling.py --root $ROOT --config $EQ --set lookback=256 $CH

## Step 4 · Kronos, lookback 256, 29 equities (GPU, ~1.6 h; SPY reuses earlier paths)

In [ ]:
!cd $CODE && python scripts/run_sampling.py --root $ROOT --kronos-path $KRONOS --config $EQ --set lookback=256

## Step 5 · ETH hourly, lookback 64: Kronos and Chronos (GPU, ~30 min)

In [ ]:
!cd $CODE && python scripts/run_sampling.py --root $ROOT --kronos-path $KRONOS --config $ETH
!cd $CODE && python scripts/run_sampling.py --root $ROOT --config $ETH $CH

## Step 6 · Reports (CPU, ~5 min) — runs with whatever steps are done; missing ones are listed

In [ ]:
!cd $CODE && python scripts/run_phaseb.py --root $ROOT --config $EQ --run kronos64 "lookback=64" --run kronos256 "lookback=256" --run chronos64 "lookback=64;model_family=chronos;model_id=amazon/chronos-t5-small;amp=false" --run chronos256 "lookback=256;model_family=chronos;model_id=amazon/chronos-t5-small;amp=false"
!cd $CODE && python scripts/run_phaseb.py --root $ROOT --config $ETH --run kronos64 "lookback=64" --run chronos64 "lookback=64;model_family=chronos;model_id=amazon/chronos-t5-small;amp=false"